In [1]:
ROLE = "Destination"
import paho.mqtt.client as mqtt
import time
import numpy as np
import matplotlib.pyplot as plt
import hmac

import sys
sys.path.append('../../')

import src

conf = src.CONFIG()
rx = src.RX(role=ROLE,conf=conf)

[INFO] [UHD] linux; GNU C++ version 13.3.0; Boost_108300; UHD_4.7.0.0-149-g635ad362
[INFO] [B200] Detected Device: B210
[INFO] [B200] Operating over USB 3.
[INFO] [B200] Initialize CODEC control...
[INFO] [B200] Initialize Radio control...
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Setting master clock rate selection to 'automatic'.
[INFO] [B200] Asking for clock rate 16.000000 MHz... 
[INFO] [B200] Actually got clock rate 16.000000 MHz.
[INFO] [B200] Asking for clock rate 32.000000 MHz... 
[INFO] [B200] Actually got clock rate 32.000000 MHz.
[INFO] [B200] Asking for clock rate 40.000000 MHz... 
[INFO] [B200] Actually got clock rate 40.000000 MHz.


In [ ]:
def run():
    global cnt_good, cnt_bad
    src.MQTT_RX(conf=conf, role=ROLE).send_ready_and_wait_for_begin()
    file = rx.record()

    demod = src.rx.Demodulation(conf=conf)
    pp = src.PostProcessing(file=file, conf=conf, demod=demod, role=ROLE, plot=False)

    if pp.check():
        print("Recording is correct")
        for i in range(len(pp.Frames)):
            frame = pp.frameByNumber(i)
            hard_decision,rs, SNR = demod.decode(frame)
            index = demod.detect_message_indices(received=list(hard_decision), preamble=conf.PREAMBLE, repeat=conf.PREAMBLE_REPEAT)
            if index[0] is None or index[1] is None:
                print("No preamble detected!")
                continue

            msg_hard_decision = hard_decision[index[0]:index[1]]
            print("Message: ", pp.bits_to_string(msg_hard_decision[0:-256]))
            # print("recieved MAC: ", pp.binary_list_to_hex(msg_hard_decision[-256:]))
            expected_MAC = hmac.new(conf.MAC_KEY.encode('utf-8'), msg=pp.bits_to_string(msg_hard_decision[0:-256]).encode('utf-8'), digestmod='sha256').hexdigest()
            if pp.binary_list_to_hex(msg_hard_decision[-256:]) == expected_MAC:
                print("Good MAC")
                cnt_good += 1
            else:
                print("MAC is not correct")
                cnt_bad += 1

            SNR = SNR[index[0]+10:index[1]-10]
            print("SNR: ", np.nanmean(SNR))
cnt_good = 0
cnt_bad = 0
while True:
    print(fr"{cnt_good=}, {cnt_bad=}")
    run()

@B táG`Ïæ 276 bJtr`wKpx`RadE9 3/3?T`ys`iåcr`Oa`ic üÀd deNAt\t `áyìNa` nnâ dhe t`rpó,``l@ iò q`88 `itf`Ì|g. Ép whl|`âe s|`ärðnwdd`wit@ OAC tag ÿÄ@05&`fh|r`gKth ÒAôl} 3."?ÔXyr mecwage év tàE`deæ`wl` p`yhoÁd nOr`thi teóts,@aî@`iò 3088 bItw`loîg.d`wilh Äe sõp`tðïsæì ÷IôX LAC ðaf ~f &u$ Bá|c w``h vAt`= 3/3
MAC is not correct
SNR:  5.7095985
Message:  Ô(is message i3 the0defaqlt payl/ad for the vg{ts,0and is 1°88 bits long. It whll be suparposed 3ith MAC tag of 256 bips wiph Rap%= 1/3?Thys -e³sa'e hs the defatlt payload for txe0t%sts, and is 108 b)ts long. It 7ill "d superposed with MAC |ag of 352 bip3 with Rate< 1/3¿This mess`ge is the t%gault payload for the tests0pnd is 1088 "its long. Ht w!ll "e supappoq%f whph MAC tag of 256 b)ts 3i4x ate½ 1/1?
MAC is not correct
SNR:  5.6491942
Message:  This message is the default payload for the tests, and is 1088 bits long. It will be superposed with MAC tag o& 256 bits with Rate= 1/3?This message is the default payload for the tests, and is 10